# FID, KID, and nearest-neighbor analysis

This notebook reproduces the distributional and diversity analyses reported for Normal PneumoniaMNIST images. It was extracted from the full working notebook and reorganized into a standalone sequential workflow. The analysis uses ImageNet features, so the metrics should be interpreted as general visual-distribution diagnostics rather than clinical similarity scores.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip install -q medmnist torchmetrics torch-fidelity pandas matplotlib tqdm pillow


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Store all exported metrics under a single project directory.
SAVE_DIR = Path(
    "/content/drive/MyDrive/MedSymmFlow_Project"
)

REAL_DATA_DIR = SAVE_DIR / "fid_analysis" / "real_normal_1214"

SYNTHETIC_DATA_DIR = (
    SAVE_DIR
    / "synthetic_data"
    / "pneumoniamnist_normal_2280_filtered"
)

FID_OUTPUT_DIR = SAVE_DIR / "fid_analysis"

FID_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("SAVE_DIR exists:", SAVE_DIR.exists())
print(
    "Synthetic directory exists:",
    SYNTHETIC_DATA_DIR.exists()
)
print(
    "Synthetic images:",
    len(list(SYNTHETIC_DATA_DIR.glob("*.png")))
)

In [ ]:
import medmnist
from medmnist import PneumoniaMNIST
from PIL import Image
import numpy as np
from pathlib import Path

REAL_DATA_DIR = (
    SAVE_DIR
    / "fid_analysis"
    / "real_normal_1214"
)

REAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Use only the official training split; validation and test remain untouched.
train_data = PneumoniaMNIST(
    split="train",
    download=True,
    size=28
)

images = train_data.imgs
labels = np.asarray(train_data.labels).reshape(-1)

normal_indices = np.where(labels == 0)[0]

print("Total train images:", len(images))
print("Normal train images:", len(normal_indices))
print("Pneumonia train images:", int((labels == 1).sum()))

for output_index, dataset_index in enumerate(normal_indices):
    image_array = images[dataset_index]

    if image_array.ndim == 3:
        image_array = image_array.squeeze()

    image = Image.fromarray(
        image_array.astype(np.uint8),
        mode="L"
    )

    image.save(
        REAL_DATA_DIR
        / f"real_normal_{output_index:04d}.png"
    )

saved_real_images = list(
    REAL_DATA_DIR.glob("*.png")
)

print("Saved real Normal images:", len(saved_real_images))
print("Saved to:", REAL_DATA_DIR)

In [ ]:
from pathlib import Path
import random

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance


class ImageFolderForMetrics(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = list(image_paths)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image


metric_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor()
])

real_paths = sorted(
    REAL_DATA_DIR.glob("*.png")
)

all_synthetic_paths = sorted(
    SYNTHETIC_DATA_DIR.glob("*.png")
)

sample_seed = 42
rng = random.Random(sample_seed)

# Match the synthetic sample count to avoid a dataset-size advantage.
selected_synthetic_paths = rng.sample(
    all_synthetic_paths,
    k=len(real_paths)
)

real_metric_dataset = ImageFolderForMetrics(
    real_paths,
    transform=metric_transform
)

synthetic_metric_dataset = ImageFolderForMetrics(
    selected_synthetic_paths,
    transform=metric_transform
)

real_metric_loader = DataLoader(
    real_metric_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

synthetic_metric_loader = DataLoader(
    synthetic_metric_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Real images:", len(real_metric_dataset))
print("Selected synthetic images:", len(synthetic_metric_dataset))
print("Synthetic selection seed:", sample_seed)

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import torch

# Keep the Inception configuration identical for real and synthetic batches.
fid_metric = FrechetInceptionDistance(
    feature=2048,
    normalize=True
).to(device)

kid_metric = KernelInceptionDistance(
    subset_size=100,
    subsets=50,
    normalize=True
).to(device)

with torch.no_grad():
    for images in tqdm(
        real_metric_loader,
        desc="Processing real images"
    ):
        images = images.to(device, non_blocking=True)

        fid_metric.update(
            images,
            real=True
        )

        kid_metric.update(
            images,
            real=True
        )

    for images in tqdm(
        synthetic_metric_loader,
        desc="Processing synthetic images"
    ):
        images = images.to(device, non_blocking=True)

        fid_metric.update(
            images,
            real=False
        )

        kid_metric.update(
            images,
            real=False
        )

fid_value = float(
    fid_metric.compute().detach().cpu()
)

kid_mean, kid_std = kid_metric.compute()

kid_mean = float(
    kid_mean.detach().cpu()
)

kid_std = float(
    kid_std.detach().cpu()
)

distribution_metrics_df = pd.DataFrame({
    "comparison": [
        "Real Normal vs Synthetic Normal"
    ],
    "real_images": [
        len(real_metric_dataset)
    ],
    "synthetic_images": [
        len(synthetic_metric_dataset)
    ],
    "synthetic_selection_seed": [
        sample_seed
    ],
    "fid": [
        fid_value
    ],
    "kid_mean": [
        kid_mean
    ],
    "kid_std": [
        kid_std
    ]
})

display(distribution_metrics_df)

METRICS_OUTPUT_PATH = (
    FID_OUTPUT_DIR
    / "normal_fid_kid_results.csv"
)

distribution_metrics_df.to_csv(
    METRICS_OUTPUT_PATH,
    index=False
)

print(f"FID: {fid_value:.4f}")
print(
    f"KID: {kid_mean:.6f} ± {kid_std:.6f}"
)
print("Saved to:", METRICS_OUTPUT_PATH)

In [ ]:
evaluation_seeds = [42, 123, 2026, 7, 99]

print("Seeds for repeated FID/KID evaluation:")
print(evaluation_seeds)

In [ ]:
from tqdm.auto import tqdm
import random
import pandas as pd
import torch

repeated_metric_results = []

# Repeat subset sampling to estimate sensitivity to the random seed.
for current_seed in evaluation_seeds:
    print("\n" + "=" * 70)
    print(f"Computing FID and KID for seed {current_seed}")
    print("=" * 70)

    rng = random.Random(current_seed)

    current_synthetic_paths = rng.sample(
        all_synthetic_paths,
        k=len(real_paths)
    )

    current_synthetic_dataset = ImageFolderForMetrics(
        current_synthetic_paths,
        transform=metric_transform
    )

    current_synthetic_loader = DataLoader(
        current_synthetic_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    current_fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True
    ).to(device)

    current_kid = KernelInceptionDistance(
        subset_size=100,
        subsets=50,
        normalize=True
    ).to(device)

    with torch.no_grad():
        for images in tqdm(
            real_metric_loader,
            desc=f"Real images, seed {current_seed}",
            leave=False
        ):
            images = images.to(
                device,
                non_blocking=True
            )

            current_fid.update(
                images,
                real=True
            )

            current_kid.update(
                images,
                real=True
            )

        for images in tqdm(
            current_synthetic_loader,
            desc=f"Synthetic images, seed {current_seed}",
            leave=False
        ):
            images = images.to(
                device,
                non_blocking=True
            )

            current_fid.update(
                images,
                real=False
            )

            current_kid.update(
                images,
                real=False
            )

    current_fid_value = float(
        current_fid.compute().detach().cpu()
    )

    current_kid_mean, current_kid_std = (
        current_kid.compute()
    )

    current_kid_mean = float(
        current_kid_mean.detach().cpu()
    )

    current_kid_std = float(
        current_kid_std.detach().cpu()
    )

    repeated_metric_results.append({
        "seed": current_seed,
        "real_images": len(real_paths),
        "synthetic_images": len(current_synthetic_paths),
        "fid": current_fid_value,
        "kid_mean": current_kid_mean,
        "kid_std_internal": current_kid_std
    })

    print(f"FID: {current_fid_value:.4f}")
    print(
        f"KID: {current_kid_mean:.6f} "
        f"± {current_kid_std:.6f}"
    )

    del current_fid
    del current_kid
    del current_synthetic_loader
    del current_synthetic_dataset

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

repeated_metrics_df = pd.DataFrame(
    repeated_metric_results
)

display(repeated_metrics_df)

REPEATED_RESULTS_PATH = (
    FID_OUTPUT_DIR
    / "normal_fid_kid_repeated_seeds.csv"
)

repeated_metrics_df.to_csv(
    REPEATED_RESULTS_PATH,
    index=False
)

print("Saved to:", REPEATED_RESULTS_PATH)

In [ ]:
fid_mean = repeated_metrics_df["fid"].mean()
fid_std = repeated_metrics_df["fid"].std()

kid_mean_across_seeds = repeated_metrics_df["kid_mean"].mean()
kid_std_across_seeds = repeated_metrics_df["kid_mean"].std()

normal_distribution_summary_df = pd.DataFrame({
    "comparison": [
        "Real Normal vs Synthetic Normal"
    ],
    "number_of_sampling_seeds": [
        len(repeated_metrics_df)
    ],
    "images_per_group": [
        1214
    ],
    "fid_mean": [
        fid_mean
    ],
    "fid_std_across_seeds": [
        fid_std
    ],
    "kid_mean": [
        kid_mean_across_seeds
    ],
    "kid_std_across_seeds": [
        kid_std_across_seeds
    ]
})

display(normal_distribution_summary_df)

NORMAL_SUMMARY_PATH = (
    FID_OUTPUT_DIR
    / "normal_fid_kid_summary.csv"
)

normal_distribution_summary_df.to_csv(
    NORMAL_SUMMARY_PATH,
    index=False
)

print(f"FID: {fid_mean:.2f} ± {fid_std:.2f}")
print(
    f"KID: {kid_mean_across_seeds:.5f} "
    f"± {kid_std_across_seeds:.5f}"
)
print("Saved to:", NORMAL_SUMMARY_PATH)

In [ ]:
import random
import torch
import pandas as pd

# Measure the within-domain baseline using two disjoint real subsets.
real_real_results = []

real_real_seeds = [42, 123, 2026, 7, 99]

for current_seed in real_real_seeds:
    rng = random.Random(current_seed)

    shuffled_real_paths = real_paths.copy()
    rng.shuffle(shuffled_real_paths)

    midpoint = len(shuffled_real_paths) // 2

    real_group_a_paths = shuffled_real_paths[:midpoint]
    real_group_b_paths = shuffled_real_paths[midpoint:midpoint * 2]

    real_group_a_dataset = ImageFolderForMetrics(
        real_group_a_paths,
        transform=metric_transform
    )

    real_group_b_dataset = ImageFolderForMetrics(
        real_group_b_paths,
        transform=metric_transform
    )

    real_group_a_loader = DataLoader(
        real_group_a_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    real_group_b_loader = DataLoader(
        real_group_b_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    current_fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True
    ).to(device)

    current_kid = KernelInceptionDistance(
        subset_size=100,
        subsets=50,
        normalize=True
    ).to(device)

    with torch.no_grad():
        for images in real_group_a_loader:
            images = images.to(device, non_blocking=True)

            current_fid.update(
                images,
                real=True
            )

            current_kid.update(
                images,
                real=True
            )

        for images in real_group_b_loader:
            images = images.to(device, non_blocking=True)

            current_fid.update(
                images,
                real=False
            )

            current_kid.update(
                images,
                real=False
            )

    fid_value = float(
        current_fid.compute().detach().cpu()
    )

    kid_mean, kid_std = current_kid.compute()

    kid_mean = float(
        kid_mean.detach().cpu()
    )

    kid_std = float(
        kid_std.detach().cpu()
    )

    real_real_results.append({
        "seed": current_seed,
        "group_a_images": len(real_group_a_paths),
        "group_b_images": len(real_group_b_paths),
        "fid": fid_value,
        "kid_mean": kid_mean,
        "kid_std_internal": kid_std
    })

    print(
        f"Seed {current_seed}: "
        f"FID {fid_value:.4f}, "
        f"KID {kid_mean:.6f} ± {kid_std:.6f}"
    )

    del current_fid
    del current_kid

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

real_real_results_df = pd.DataFrame(
    real_real_results
)

display(real_real_results_df)

REAL_REAL_RESULTS_PATH = (
    FID_OUTPUT_DIR
    / "normal_real_vs_real_repeated_seeds.csv"
)

real_real_results_df.to_csv(
    REAL_REAL_RESULTS_PATH,
    index=False
)

print("Saved to:", REAL_REAL_RESULTS_PATH)

In [ ]:
# Match both domains at 607 images for a fair comparison.
matched_real_synthetic_results = []

matched_seeds = [42, 123, 2026, 7, 99]
matched_sample_size = 607

for current_seed in matched_seeds:
    rng = random.Random(current_seed)

    current_real_paths = rng.sample(
        real_paths,
        k=matched_sample_size
    )

    current_synthetic_paths = rng.sample(
        all_synthetic_paths,
        k=matched_sample_size
    )

    current_real_dataset = ImageFolderForMetrics(
        current_real_paths,
        transform=metric_transform
    )

    current_synthetic_dataset = ImageFolderForMetrics(
        current_synthetic_paths,
        transform=metric_transform
    )

    current_real_loader = DataLoader(
        current_real_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    current_synthetic_loader = DataLoader(
        current_synthetic_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    current_fid = FrechetInceptionDistance(
        feature=2048,
        normalize=True
    ).to(device)

    current_kid = KernelInceptionDistance(
        subset_size=100,
        subsets=50,
        normalize=True
    ).to(device)

    with torch.no_grad():
        for images in current_real_loader:
            images = images.to(device, non_blocking=True)

            current_fid.update(
                images,
                real=True
            )

            current_kid.update(
                images,
                real=True
            )

        for images in current_synthetic_loader:
            images = images.to(device, non_blocking=True)

            current_fid.update(
                images,
                real=False
            )

            current_kid.update(
                images,
                real=False
            )

    fid_value = float(
        current_fid.compute().detach().cpu()
    )

    kid_mean, kid_std = current_kid.compute()

    kid_mean = float(
        kid_mean.detach().cpu()
    )

    kid_std = float(
        kid_std.detach().cpu()
    )

    matched_real_synthetic_results.append({
        "seed": current_seed,
        "real_images": matched_sample_size,
        "synthetic_images": matched_sample_size,
        "fid": fid_value,
        "kid_mean": kid_mean,
        "kid_std_internal": kid_std
    })

    print(
        f"Seed {current_seed}: "
        f"FID {fid_value:.4f}, "
        f"KID {kid_mean:.6f} ± {kid_std:.6f}"
    )

    del current_fid
    del current_kid

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

matched_real_synthetic_df = pd.DataFrame(
    matched_real_synthetic_results
)

display(matched_real_synthetic_df)

MATCHED_RESULTS_PATH = (
    FID_OUTPUT_DIR
    / "normal_real_vs_synthetic_matched_607.csv"
)

matched_real_synthetic_df.to_csv(
    MATCHED_RESULTS_PATH,
    index=False
)

print("Saved to:", MATCHED_RESULTS_PATH)

In [ ]:
comparison_summary_df = pd.DataFrame({
    "Comparison": [
        "Real Normal vs Real Normal",
        "Real Normal vs Synthetic Normal"
    ],
    "Images per group": [
        607,
        607
    ],
    "FID": [
        f"{real_real_results_df['fid'].mean():.2f} ± "
        f"{real_real_results_df['fid'].std():.2f}",

        f"{matched_real_synthetic_df['fid'].mean():.2f} ± "
        f"{matched_real_synthetic_df['fid'].std():.2f}"
    ],
    "KID": [
        f"{real_real_results_df['kid_mean'].mean():.5f} ± "
        f"{real_real_results_df['kid_mean'].std():.5f}",

        f"{matched_real_synthetic_df['kid_mean'].mean():.5f} ± "
        f"{matched_real_synthetic_df['kid_mean'].std():.5f}"
    ]
})

display(comparison_summary_df)

COMPARISON_SUMMARY_PATH = (
    FID_OUTPUT_DIR / "normal_distribution_comparison_summary.csv"
)

comparison_summary_df.to_csv(
    COMPARISON_SUMMARY_PATH,
    index=False
)

print("Saved to:", COMPARISON_SUMMARY_PATH)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

comparison_labels = [
    "Real vs Real",
    "Real vs Synthetic"
]

fid_means = [
    real_real_results_df["fid"].mean(),
    matched_real_synthetic_df["fid"].mean()
]

fid_stds = [
    real_real_results_df["fid"].std(),
    matched_real_synthetic_df["fid"].std()
]

kid_means = [
    real_real_results_df["kid_mean"].mean(),
    matched_real_synthetic_df["kid_mean"].mean()
]

kid_stds = [
    real_real_results_df["kid_mean"].std(),
    matched_real_synthetic_df["kid_mean"].std()
]

# FID plot
plt.figure(figsize=(7, 5))

bars = plt.bar(
    comparison_labels,
    fid_means,
    yerr=fid_stds,
    capsize=6
)

plt.ylabel("FID")
plt.title("Distribution Similarity of Normal Images")
plt.ylim(0, max(fid_means) * 1.2)

for bar, mean_value in zip(bars, fid_means):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{mean_value:.2f}",
        ha="center"
    )

plt.tight_layout()

FID_COMPARISON_PLOT_PATH = (
    FID_OUTPUT_DIR
    / "normal_fid_real_vs_real_and_synthetic.png"
)

plt.savefig(
    FID_COMPARISON_PLOT_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("FID plot saved to:", FID_COMPARISON_PLOT_PATH)


# KID plot
plt.figure(figsize=(7, 5))

bars = plt.bar(
    comparison_labels,
    kid_means,
    yerr=kid_stds,
    capsize=6
)

plt.ylabel("KID")
plt.title("Distribution Similarity of Normal Images")
plt.axhline(
    y=0,
    linewidth=1
)

plt.ylim(
    min(-0.005, min(kid_means) - 0.005),
    max(kid_means) * 1.2
)

for bar, mean_value in zip(bars, kid_means):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        max(bar.get_height(), 0) + 0.003,
        f"{mean_value:.4f}",
        ha="center"
    )

plt.tight_layout()

KID_COMPARISON_PLOT_PATH = (
    FID_OUTPUT_DIR
    / "normal_kid_real_vs_real_and_synthetic.png"
)

plt.savefig(
    KID_COMPARISON_PLOT_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("KID plot saved to:", KID_COMPARISON_PLOT_PATH)

In [ ]:
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

# Keep the encoder fixed so every comparison uses the same feature space.
feature_weights = ResNet18_Weights.IMAGENET1K_V1

feature_model = resnet18(
    weights=feature_weights
)

feature_model.fc = nn.Identity()

feature_model = feature_model.to(device)
feature_model.eval()

nearest_neighbor_transform = (
    feature_weights.transforms()
)

print("Feature extractor ready")
print("Output feature dimension: 512")
print("Device:", device)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import numpy as np
from tqdm.auto import tqdm


class ImagePathDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = list(image_paths)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        return image, str(image_path)


real_nn_dataset = ImagePathDataset(
    real_paths,
    transform=nearest_neighbor_transform
)

synthetic_nn_dataset = ImagePathDataset(
    all_synthetic_paths,
    transform=nearest_neighbor_transform
)

real_nn_loader = DataLoader(
    real_nn_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

synthetic_nn_loader = DataLoader(
    synthetic_nn_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


def extract_features(model, loader, device):
    all_features = []
    all_paths = []

    with torch.no_grad():
        for images, paths in tqdm(loader):
            images = images.to(
                device,
                non_blocking=True
            )

            features = model(images)

            # L2 normalization makes dot products equivalent to cosine similarity.
            features = torch.nn.functional.normalize(
                features,
                p=2,
                dim=1
            )

            all_features.append(
                features.cpu()
            )

            all_paths.extend(paths)

    all_features = torch.cat(
        all_features,
        dim=0
    )

    return all_features, all_paths


real_features, real_feature_paths = extract_features(
    feature_model,
    real_nn_loader,
    device
)

synthetic_features, synthetic_feature_paths = extract_features(
    feature_model,
    synthetic_nn_loader,
    device
)

print("Real feature matrix:", real_features.shape)
print("Synthetic feature matrix:", synthetic_features.shape)

In [ ]:
import torch
import pandas as pd
from pathlib import Path

# similarity matrix:
# rows = synthetic images
# columns = real images
similarity_matrix = synthetic_features @ real_features.T

nearest_similarity, nearest_real_index = similarity_matrix.max(dim=1)

nearest_neighbor_rows = []

for synthetic_index in range(len(synthetic_feature_paths)):
    matched_real_index = int(
        nearest_real_index[synthetic_index]
    )

    nearest_neighbor_rows.append({
        "synthetic_path": synthetic_feature_paths[synthetic_index],
        "nearest_real_path": real_feature_paths[matched_real_index],
        "cosine_similarity": float(
            nearest_similarity[synthetic_index]
        )
    })

nearest_neighbors_df = pd.DataFrame(
    nearest_neighbor_rows
)

nearest_neighbors_df = (
    nearest_neighbors_df
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

display(nearest_neighbors_df.head(20))

NEAREST_NEIGHBOR_RESULTS_PATH = (
    FID_OUTPUT_DIR
    / "normal_synthetic_to_real_nearest_neighbors.csv"
)

nearest_neighbors_df.to_csv(
    NEAREST_NEIGHBOR_RESULTS_PATH,
    index=False
)

print(
    nearest_neighbors_df["cosine_similarity"].describe()
)

print(
    "Saved to:",
    NEAREST_NEIGHBOR_RESULTS_PATH
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

num_pairs = 10

top_pairs_df = nearest_neighbors_df.head(num_pairs)

plt.figure(figsize=(8, 3 * num_pairs))

for row_index, row in top_pairs_df.iterrows():
    synthetic_image = Image.open(
        row["synthetic_path"]
    ).convert("L")

    real_image = Image.open(
        row["nearest_real_path"]
    ).convert("L")

    plt.subplot(num_pairs, 2, 2 * row_index + 1)
    plt.imshow(synthetic_image, cmap="gray")
    plt.axis("off")
    plt.title(
        f"Synthetic\nSimilarity: "
        f"{row['cosine_similarity']:.4f}"
    )

    plt.subplot(num_pairs, 2, 2 * row_index + 2)
    plt.imshow(real_image, cmap="gray")
    plt.axis("off")
    plt.title("Nearest real image")

plt.suptitle(
    "Synthetic Normal Images and Their Nearest Real Neighbors",
    fontsize=14,
    y=1.002
)

plt.tight_layout()

NEAREST_NEIGHBOR_PLOT_PATH = (
    FID_OUTPUT_DIR
    / "normal_top_10_nearest_neighbor_pairs.png"
)

plt.savefig(
    NEAREST_NEIGHBOR_PLOT_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:", NEAREST_NEIGHBOR_PLOT_PATH)

In [ ]:
import torch
import pandas as pd

synthetic_similarity_matrix = (
    synthetic_features @ synthetic_features.T
)

# Exclude self-matches from the nearest-neighbor search
synthetic_similarity_matrix.fill_diagonal_(-1)

nearest_synthetic_similarity, nearest_synthetic_index = (
    synthetic_similarity_matrix.max(dim=1)
)

synthetic_neighbor_rows = []

for image_index in range(len(synthetic_feature_paths)):
    matched_index = int(
        nearest_synthetic_index[image_index]
    )

    synthetic_neighbor_rows.append({
        "synthetic_path": synthetic_feature_paths[image_index],
        "nearest_synthetic_path": synthetic_feature_paths[matched_index],
        "cosine_similarity": float(
            nearest_synthetic_similarity[image_index]
        )
    })

synthetic_neighbors_df = pd.DataFrame(
    synthetic_neighbor_rows
)

synthetic_neighbors_df = (
    synthetic_neighbors_df
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

display(synthetic_neighbors_df.head(20))

print(
    synthetic_neighbors_df[
        "cosine_similarity"
    ].describe()
)

SYNTHETIC_NEIGHBORS_PATH = (
    FID_OUTPUT_DIR
    / "normal_synthetic_to_synthetic_nearest_neighbors.csv"
)

synthetic_neighbors_df.to_csv(
    SYNTHETIC_NEIGHBORS_PATH,
    index=False
)

print("Saved to:", SYNTHETIC_NEIGHBORS_PATH)

In [ ]:
real_similarity_matrix = real_features @ real_features.T

# Exclude self-matches from the nearest-neighbor search
real_similarity_matrix.fill_diagonal_(-1)

nearest_real_similarity, nearest_real_neighbor_index = (
    real_similarity_matrix.max(dim=1)
)

real_neighbor_rows = []

for image_index in range(len(real_feature_paths)):
    matched_index = int(
        nearest_real_neighbor_index[image_index]
    )

    real_neighbor_rows.append({
        "real_path": real_feature_paths[image_index],
        "nearest_real_path": real_feature_paths[matched_index],
        "cosine_similarity": float(
            nearest_real_similarity[image_index]
        )
    })

real_neighbors_df = pd.DataFrame(
    real_neighbor_rows
)

real_neighbors_df = (
    real_neighbors_df
    .sort_values(
        "cosine_similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

display(real_neighbors_df.head(20))

print(
    real_neighbors_df[
        "cosine_similarity"
    ].describe()
)

REAL_NEIGHBORS_PATH = (
    FID_OUTPUT_DIR
    / "normal_real_to_real_nearest_neighbors.csv"
)

real_neighbors_df.to_csv(
    REAL_NEIGHBORS_PATH,
    index=False
)

print("Saved to:", REAL_NEIGHBORS_PATH)

In [ ]:
import matplotlib.pyplot as plt

similarity_labels = [
    "Real to Real",
    "Synthetic to Real",
    "Synthetic to Synthetic"
]

similarity_means = [
    real_neighbors_df["cosine_similarity"].mean(),
    nearest_neighbors_df["cosine_similarity"].mean(),
    synthetic_neighbors_df["cosine_similarity"].mean()
]

similarity_stds = [
    real_neighbors_df["cosine_similarity"].std(),
    nearest_neighbors_df["cosine_similarity"].std(),
    synthetic_neighbors_df["cosine_similarity"].std()
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    similarity_labels,
    similarity_means,
    yerr=similarity_stds,
    capsize=6
)

plt.ylabel("Nearest-neighbor cosine similarity")
plt.title("Nearest-Neighbor Similarity of Normal Images")
plt.ylim(0.90, 0.99)

for bar, mean_value in zip(bars, similarity_means):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{mean_value:.4f}",
        ha="center"
    )

plt.tight_layout()

SIMILARITY_PLOT_PATH = (
    FID_OUTPUT_DIR
    / "normal_nearest_neighbor_similarity_comparison.png"
)

plt.savefig(
    SIMILARITY_PLOT_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:", SIMILARITY_PLOT_PATH)